# MSI SEAL: local seed-0 baseline

Select a local **Python 3 kernel → Run All**. The current kernel starts computation and displays results. First use creates an independent **Python 3.11 + PyTorch 2.4 CPU** environment; no kernel switch is required.

Pipeline: `MSI molecular graph + existing node2vec features → one-hop enclosing subgraphs → DRNL labels + node-type features → DGCNN → ten-fold evaluation`.

Following DREAMwalk **Supplementary Method 1**, use **128D node2vec + 4D node-type features** and validation **patience=10**. Other architecture/training defaults follow pinned SEAL/DGCNN sources. The CPU port retains their row-normalized convolution; original sources remain in `vendor/`.

Reuse the molecular graph, 5,926 positives, 5,926 saved seed-0 negatives and node2vec fold IDs. SEAL consumes node2vec features, so this is not a walk-free input pipeline. SEAL itself has no `pp` or `τ` and needs no MeSH/ATC.

New outputs go into `04.SEAL_reproduction/`. Training starts when you run its cells; published outputs are cleared.


## 1. Directories and run name

Set `REPOSITORY_OVERRIDE` to the repository root or `00.Reproduction/` only if needed. `PYTHON_OVERRIDE` may select a Python 3.11 interpreter satisfying the pins. Use a new run name after changing settings, inputs or implementation.


In [2]:
from pathlib import Path
import ast
import csv
import html
import json
import os
import shutil
import signal
import subprocess
import sys
from IPython.display import display, HTML

RUN_NAME = 'msi_seed0'
REPOSITORY_OVERRIDE = None
PYTHON_OVERRIDE = None

def find_reproduction_root(start):
    start = Path(start).expanduser().resolve()
    for parent in [start, *start.parents]:
        for candidate in [parent/'00.Reproduction', parent]:
            if ((candidate/'00.DREAMwalk_reproduction').is_dir()
                    and (candidate/'04.SEAL_reproduction').is_dir()):
                return candidate
    raise FileNotFoundError('00.Reproduction not found; set REPOSITORY_OVERRIDE.')

REPRODUCTION = find_reproduction_root(REPOSITORY_OVERRIDE or Path.cwd())
ROOT = REPRODUCTION/'04.SEAL_reproduction'
SOURCE = REPRODUCTION/'00.DREAMwalk_reproduction'
NODE2VEC = REPRODUCTION/'01.node2vec_reproduction'
RUNNER = ROOT/'src/run_seal.py'
ENV_PREFIX = ROOT/'environment/.conda-env'
PYTHON = Path(PYTHON_OVERRIDE).expanduser().resolve() if PYTHON_OVERRIDE else ENV_PREFIX/'bin/python'
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME accepts only letters, digits, underscores and hyphens.')
RESULTS = ROOT/'results'/RUN_NAME
LOGS = ROOT/'logs'/RUN_NAME
LOGS.mkdir(parents=True, exist_ok=True)
required = [SOURCE/'data/processed/msi_graph.tsv', SOURCE/'data/processed/msi_nodetypes.tsv',
            SOURCE/'data/processed/msi_positive_pairs.tsv', SOURCE/'results/msi_repeat0_negative_pairs.csv',
            NODE2VEC/'data/msi_seed0/msi_node2vec_seed0.pkl',
            NODE2VEC/'results/msi_seed0/fold_assignments.csv',
            NODE2VEC/'results/msi_seed0/embedding_complete.json',
            NODE2VEC/'results/msi_seed0/run_config.json', RUNNER]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Required input missing: {path}')
print('SEAL project: ', ROOT)
print('Computation interpreter: ', PYTHON)
print('Results: ', RESULTS)
print('Reused node2vec: ', NODE2VEC/'data/msi_seed0/msi_node2vec_seed0.pkl')

SEAL project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction
Computation interpreter: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python
Results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/results/msi_seed0
Reused node2vec: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/01.node2vec_reproduction/data/msi_seed0/msi_node2vec_seed0.pkl


## 2. Independent CPU environment and execution helpers

Python 3.11; PyTorch 2.4.0; NumPy 1.26.2; SciPy 1.11.4; scikit-learn 1.3.2. First installation requires network access. Use CPU/four threads, without CUDA, MPS or the legacy SEAL C++ extension.

This section prepares the environment only. Interrupting a computation cell terminates its subprocess group while preserving completed subgraphs, epochs and folds.


In [3]:
def stream_command(command, log_path, env=None):
    command = list(map(str, command))
    print('Starting: ', ' '.join(command), flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\nCOMMAND: ' + repr(command) + '\n')
        with subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1,
                              start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
                raise
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)

def runner_literal(name):
    tree = ast.parse(RUNNER.read_text())
    for statement in tree.body:
        if isinstance(statement, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in statement.targets):
            return ast.literal_eval(statement.value)
    raise KeyError(name)

PINS = runner_literal('PINS')
probe = (
    'import sys; from importlib.metadata import version; '
    'import torch, numpy, scipy, sklearn; '
    'assert sys.version_info[:2] == (3,11); '
    f'expected={PINS!r}; '
    'actual={k:version(k) for k in expected}; '
    'assert actual == expected, actual; print(sys.version); print(actual)'
)
ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True) if PYTHON.is_file() else None
if ready is None or ready.returncode:
    if PYTHON_OVERRIDE:
        if ready is not None:
            print(ready.stdout, ready.stderr)
        raise RuntimeError('PYTHON_OVERRIDE does not match package pins; clear it to create a dedicated environment.')
    candidates = [shutil.which('mamba'), shutil.which('conda'),
                  str(Path.home()/'mambaforge/bin/mamba'),
                  str(Path.home()/'miniforge3/bin/mamba'),
                  str(Path.home()/'miniforge3/bin/conda'),
                  str(Path.home()/'miniconda3/bin/conda')]
    manager = next((p for p in candidates if p and Path(p).is_file()), None)
    if manager is None:
        raise RuntimeError('mamba/conda not found; install Miniforge and rerun this section.')
    install_env = dict(os.environ)
    install_env['CONDA_PKGS_DIRS'] = str(ROOT/'environment/.conda-pkgs')
    action = 'install' if (ENV_PREFIX/'conda-meta/history').is_file() else 'create'
    stream_command([manager, action, '--yes', '--prefix', ENV_PREFIX,
                    '--override-channels', '-c', 'conda-forge', 'python=3.11', 'pip'],
                   LOGS/'environment.log', env=install_env)
    stream_command([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check',
                    '-r', ROOT/'environment/requirements.txt'], LOGS/'environment.log')
    ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True)
    if ready.returncode:
        print(ready.stdout, ready.stderr)
        raise RuntimeError('Environment is not ready; inspect the installation log.')
print(ready.stdout)

CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONUNBUFFERED'] = '1'
CHILD_ENV['PYTHONDONTWRITEBYTECODE'] = '1'

def run_phase(phase, fold=None):
    command = [PYTHON, '-u', RUNNER, phase, '--run-name', RUN_NAME]
    if fold is not None:
        command += ['--fold', str(fold)]
    suffix = f'_fold{fold:02d}' if fold is not None else ''
    stream_command(command, LOGS/f'{phase}{suffix}.log', env=CHILD_ENV)

def show_rows(rows):
    if not rows:
        return
    keys = list(rows[0])
    head = ''.join('<th style="text-align:left;padding:6px">'+html.escape(str(k))+'</th>' for k in keys)
    body = ''.join('<tr>'+''.join('<td style="padding:6px">'+html.escape(str(r.get(k, '')))+'</td>' for k in keys)+'</tr>' for r in rows)
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))

def show_csv(path, tail=None):
    if not path.is_file():
        print('Not generated yet: ', path)
        return
    with path.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    show_rows(rows[-tail:] if tail else rows)

Launching: /Users/wangyuheng/mambaforge/condabin/mamba create --yes --prefix /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env --override-channels -c conda-forge python=3.11 pip

Looking for: ['python=3.11', 'pip']

Transaction

  Prefix: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env

  Updating specs:

   - python=3.11
   - pip


  Package               Version  Build               Channel          Size
────────────────────────────────────────────────────────────────────────────
  Install:
────────────────────────────────────────────────────────────────────────────

  + libexpat              2.8.5  h6111c0a_0          conda-forge      70kB
  + libpython           3.11.17  h0677fcb_2_cpython  conda-forge       2MB
  + bzip2                 1.0.8  h4e30115_10         conda-forge     125kB
  + libffi                3.7.0  h47dc5ef_1          conda-forge      4

## 3. Fixed parameters and paper reference

| Component | Setting |
|---|---|
| Graph | Shared MSI; 29,959 nodes / 478,728 input rows; undirected simple adjacency; no DDA label edges |
| Features | Existing 128D node2vec + four node-type indicators + DRNL structural one-hot |
| Subgraphs | One hop; no per-hop node cap; remove the target edge |
| DGCNN | [32,32,32,1]; row normalization; SortPooling 60th percentile, min k=10; Conv1d 16/32; MLP 128; dropout=0.5 |
| Training | Adam; lr=1e-4; batch=50; max 50 epochs; validation-loss patience=10; CPU/four threads |
| Evaluation | Same 5,926 positives + 5,926 seed-0 negatives; saved node2vec ten-fold IDs |
| Validation | Stratified 10% of each outer training fold, seed 0; outer test data do not select epochs |
| Metrics | Accuracy, AUROC, AUPR=average precision |

**Input adapter:** reuse the audited molecular-only node2vec checkpoint. Do not call the original SEAL helper to regenerate embeddings or inject negative edges. DDA labels stay outside the graph. The complete author MSI SEAL benchmark script is unavailable; record these adapter choices explicitly.

Paper references are ten SEAL repeats from `Figure2a!A32:E41`. Local evaluation is **one seed × ten folds**, compared with SEAL's own paper baseline.


In [4]:
CONFIG = runner_literal('CONFIG')
show_rows([{'parameter': k, 'value': v} for k, v in CONFIG.items()])
print('Paper SEAL results: ')
show_csv(ROOT/'reference/paper_seal_fig2a_summary.csv')
run_phase('config')

parameter,value
seed,0
folds,10
validation_fraction,0.1
device,cpu
threads,4
hops,1
max_nodes_per_hop,None
latent_dims,"[32, 32, 32, 1]"
sortpooling_percentile,0.6
hidden_dim,128


Paper SEAL results:


metric,mean,SD,n
accuracy,0.7294390552509488,0.013069738112634448,10
auroc,0.7936975331042188,0.013848399089751065,10
aupr,0.7937583979948021,0.012199522120737346,10


Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py config --run-name msi_seed0
{
  "config": {
    "seed": 0,
    "folds": 10,
    "validation_fraction": 0.1,
    "device": "cpu",
    "threads": 4,
    "hops": 1,
    "max_nodes_per_hop": null,
    "latent_dims": [
      32,
      32,
      32,
      1
    ],
    "sortpooling_percentile": 0.6,
    "hidden_dim": 128,
    "dropout": 0.5,
    "batch_size": 50,
    "optimizer": "Adam",
    "learning_rate": 0.0001,
    "weight_decay": 0.0,
    "max_epochs": 50,
    "patience": 10,
    "selection_metric": "validation loss",
    "train_drop_last": true,
    "node2vec_dimension": 128,
    "node_types": [
      "drug",
      "disease",
      "gene",
      "etc"
    ],
    "features": "DRNL one-hot + existing 128D node2vec + 4D node-type one-hot",
    "grap

## 4. Prepare graph, node features and partitions

Check embedding/graph hashes, complete node coverage, pair order and saved fold IDs. Build 132D node features and save train/validation/test indices per fold. Inputs are read-only; outputs belong to this model directory.


In [5]:
run_phase('prepare')
info = json.loads((RESULTS/'prepare_complete.json').read_text())
show_rows([{k: info[k] for k in ['nodes','molecular_rows','simple_edges','positives','negatives','node_feature_dimension','treatment_edges_added']}])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py prepare --run-name msi_seed0
Prepared 29,959 nodes, 132D features, 11,852 labeled pairs and exact saved ten-fold IDs


nodes,molecular_rows,simple_edges,positives,negatives,node_feature_dimension,treatment_edges_added
29959,478728,478728,5926,5926,132,0


## 5. Generate one enclosing-subgraph corpus

Extract one-hop enclosing subgraphs and DRNL labels for all **11,852 pairs**. Cache them by pair ID in SQLite and read training batches from disk.

Report/save every 100 pairs and resume saved work after interruption. All folds reuse this corpus because the molecular graph contains no DDA labels and node2vec features are fixed.


In [6]:
run_phase('subgraphs')
info = json.loads((RESULTS/'subgraphs_complete.json').read_text())
show_rows([{k: info[k] for k in ['pairs','max_nodes','label_count','sortpooling_k']}])
print('Cache: ', ROOT/'data'/RUN_NAME/'subgraphs.sqlite')

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py subgraphs --run-name msi_seed0
Subgraphs 100/11,852; 0.1s
Subgraphs 200/11,852; 0.1s
Subgraphs 300/11,852; 0.2s
Subgraphs 400/11,852; 0.2s
Subgraphs 500/11,852; 0.3s
Subgraphs 600/11,852; 0.4s
Subgraphs 700/11,852; 0.4s
Subgraphs 800/11,852; 0.5s
Subgraphs 900/11,852; 0.6s
Subgraphs 1,000/11,852; 0.6s
Subgraphs 1,100/11,852; 0.7s
Subgraphs 1,200/11,852; 0.8s
Subgraphs 1,300/11,852; 0.8s
Subgraphs 1,400/11,852; 0.9s
Subgraphs 1,500/11,852; 1.0s
Subgraphs 1,600/11,852; 1.0s
Subgraphs 1,700/11,852; 1.1s
Subgraphs 1,800/11,852; 1.2s
Subgraphs 1,900/11,852; 1.2s
Subgraphs 2,000/11,852; 1.3s
Subgraphs 2,100/11,852; 1.3s
Subgraphs 2,200/11,852; 1.4s
Subgraphs 2,300/11,852; 1.5s
Subgraphs 2,400/11,852; 1.5s
Subgraphs 2,500/11,852; 1.6s
Subgraphs 2,600/11

pairs,max_nodes,label_count,sortpooling_k
11852,985,75,31


Cache: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/data/msi_seed0/subgraphs.sqlite


## 6. Ten-fold DGCNN training and held-out evaluation

Initialize DGCNN separately per fold. Choose the best epoch by **validation loss**, stopping after ten non-improving epochs or a maximum of 50.

Save each improved model as `best.pt`; save `last.pt`, optimizer and RNG states every epoch. Predict the test fold only with the selected best model. Reuse completed folds and resume incomplete folds from the last epoch.

The next ten cells run one fold each. Logs use `logs/<RUN_NAME>/train_foldXX.log`.


### 6.1 Fold 1/10

In [7]:
run_phase('train', fold=1)
show_csv(RESULTS/'fold01'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold01'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 1
Fold 1/10 | epoch 1 | training batch 20 | loss 0.69834
Fold 1/10 | epoch 1 | training batch 40 | loss 0.69452
Fold 1/10 | epoch 1 | training batch 60 | loss 0.69076
Fold 1/10 | epoch 1 | training batch 80 | loss 0.68874
Fold 1/10 | epoch 1 | training batch 100 | loss 0.68515
Fold 1/10 | epoch 1 | training batch 120 | loss 0.68385
Fold 1/10 | epoch 1 | training batch 140 | loss 0.68359
Fold 1/10 | epoch 1 | training batch 160 | loss 0.68176
Fold 1/10 | epoch 1 | training batch 180 | loss 0.67999
Fold 1/10 | epoch 1/50 | train 0.67893 | validation 0.65972 | best 1 | patience 0/10 | 5.0s
Fold 1/10 | epoch 2 | training batch 20 | loss 0.67453
Fold 1/10 | epoch 2 | training batch 40 | loss 0.66572
Fold 1/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
34,0.5038103843546663,9550,0.5755930921130908,0.7853556647061717,26,8
35,0.4994642431823371,9550,0.5655296633147479,0.7848075693375776,26,9
36,0.49664425662674827,9550,0.5675026228635693,0.7858053839829668,26,10


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
1,0.7166947723440135,0.7849162090607397,0.8002599734555893,0.5597959087429722,26,0.563624073661368,9599,1067,1186


### 6.2 Fold 2/10

In [8]:
run_phase('train', fold=2)
show_csv(RESULTS/'fold02'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold02'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 2
Fold 2/10 | epoch 1 | training batch 20 | loss 0.69921
Fold 2/10 | epoch 1 | training batch 40 | loss 0.69428
Fold 2/10 | epoch 1 | training batch 60 | loss 0.69108
Fold 2/10 | epoch 1 | training batch 80 | loss 0.68915
Fold 2/10 | epoch 1 | training batch 100 | loss 0.68625
Fold 2/10 | epoch 1 | training batch 120 | loss 0.68428
Fold 2/10 | epoch 1 | training batch 140 | loss 0.68136
Fold 2/10 | epoch 1 | training batch 160 | loss 0.67867
Fold 2/10 | epoch 1 | training batch 180 | loss 0.67727
Fold 2/10 | epoch 1/50 | train 0.67664 | validation 0.66537 | best 1 | patience 0/10 | 5.6s
Fold 2/10 | epoch 2 | training batch 20 | loss 0.66798
Fold 2/10 | epoch 2 | training batch 40 | loss 0.66269
Fold 2/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
48,0.4689102965499718,9550,0.5670080774838162,0.7903956826949428,42,6
49,0.46781441179245553,9550,0.5650924684404768,0.7916218704105796,42,7
50,0.46508406235285454,9550,0.5663992444711415,0.7887232891343607,42,8


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
2,0.7335581787521079,0.8046589070351403,0.8126740565187143,0.5416197495355975,42,0.5628382853514252,9599,1067,1186


### 6.3 Fold 3/10

In [9]:
run_phase('train', fold=3)
show_csv(RESULTS/'fold03'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold03'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 3
Fold 3/10 | epoch 1 | training batch 20 | loss 0.69709
Fold 3/10 | epoch 1 | training batch 40 | loss 0.69187
Fold 3/10 | epoch 1 | training batch 60 | loss 0.69011
Fold 3/10 | epoch 1 | training batch 80 | loss 0.68850
Fold 3/10 | epoch 1 | training batch 100 | loss 0.68577
Fold 3/10 | epoch 1 | training batch 120 | loss 0.68425
Fold 3/10 | epoch 1 | training batch 140 | loss 0.68120
Fold 3/10 | epoch 1 | training batch 160 | loss 0.67976
Fold 3/10 | epoch 1 | training batch 180 | loss 0.67748
Fold 3/10 | epoch 1/50 | train 0.67645 | validation 0.65492 | best 1 | patience 0/10 | 4.8s
Fold 3/10 | epoch 2 | training batch 20 | loss 0.66124
Fold 3/10 | epoch 2 | training batch 40 | loss 0.65778
Fold 3/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
38,0.49199032395457226,9600,0.543288539253671,0.8044406265151675,30,8
39,0.4907387951388955,9600,0.5473001760752713,0.797946047740512,30,9
40,0.4873056827733914,9600,0.5428382583947079,0.8020075749590685,30,10


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
3,0.7265822784810126,0.7954884690761588,0.7984583187435125,0.5477907719994396,30,0.5388681810373666,9600,1067,1185


### 6.4 Fold 4/10

In [10]:
run_phase('train', fold=4)
show_csv(RESULTS/'fold04'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold04'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 4
Fold 4/10 | epoch 1 | training batch 20 | loss 0.69654
Fold 4/10 | epoch 1 | training batch 40 | loss 0.69301
Fold 4/10 | epoch 1 | training batch 60 | loss 0.69099
Fold 4/10 | epoch 1 | training batch 80 | loss 0.68849
Fold 4/10 | epoch 1 | training batch 100 | loss 0.68450
Fold 4/10 | epoch 1 | training batch 120 | loss 0.68131
Fold 4/10 | epoch 1 | training batch 140 | loss 0.67946
Fold 4/10 | epoch 1 | training batch 160 | loss 0.67826
Fold 4/10 | epoch 1 | training batch 180 | loss 0.67482
Fold 4/10 | epoch 1/50 | train 0.67411 | validation 0.67856 | best 1 | patience 0/10 | 4.8s
Fold 4/10 | epoch 2 | training batch 20 | loss 0.66294
Fold 4/10 | epoch 2 | training batch 40 | loss 0.66521
Fold 4/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
37,0.49131701545168954,9600,0.6097441586581144,0.7535977542143615,29,8
38,0.4898735809450348,9600,0.6087109620256075,0.7547729971681739,29,9
39,0.4868782164218525,9600,0.6135812802301351,0.7538226138527591,29,10


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
4,0.7071729957805907,0.7774087325099129,0.7803828818155927,0.57069426749829,29,0.6060429361230766,9600,1067,1185


### 6.5 Fold 5/10

In [11]:
run_phase('train', fold=5)
show_csv(RESULTS/'fold05'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold05'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 5
Fold 5/10 | epoch 1 | training batch 20 | loss 0.69744
Fold 5/10 | epoch 1 | training batch 40 | loss 0.69301
Fold 5/10 | epoch 1 | training batch 60 | loss 0.69139
Fold 5/10 | epoch 1 | training batch 80 | loss 0.68782
Fold 5/10 | epoch 1 | training batch 100 | loss 0.68421
Fold 5/10 | epoch 1 | training batch 120 | loss 0.68262
Fold 5/10 | epoch 1 | training batch 140 | loss 0.68027
Fold 5/10 | epoch 1 | training batch 160 | loss 0.67893
Fold 5/10 | epoch 1 | training batch 180 | loss 0.67665
Fold 5/10 | epoch 1/50 | train 0.67564 | validation 0.66114 | best 1 | patience 0/10 | 4.6s
Fold 5/10 | epoch 2 | training batch 20 | loss 0.66680
Fold 5/10 | epoch 2 | training batch 40 | loss 0.66189
Fold 5/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
48,0.4793915861907105,9600,0.5409997128501828,0.8074393405991104,45,3
49,0.47732192852223915,9600,0.5436834792947813,0.8059672126539761,45,4
50,0.47745948455606896,9600,0.5357393633012137,0.8117503214790143,50,0


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
5,0.7417721518987341,0.8014618750284855,0.8008490365534515,0.555680707134778,50,0.5357393633012137,9600,1067,1185


### 6.6 Fold 6/10

In [12]:
run_phase('train', fold=6)
show_csv(RESULTS/'fold06'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold06'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 6
Fold 6/10 | epoch 1 | training batch 20 | loss 0.69620
Fold 6/10 | epoch 1 | training batch 40 | loss 0.69233
Fold 6/10 | epoch 1 | training batch 60 | loss 0.68972
Fold 6/10 | epoch 1 | training batch 80 | loss 0.68666
Fold 6/10 | epoch 1 | training batch 100 | loss 0.68366
Fold 6/10 | epoch 1 | training batch 120 | loss 0.68289
Fold 6/10 | epoch 1 | training batch 140 | loss 0.68141
Fold 6/10 | epoch 1 | training batch 160 | loss 0.68022
Fold 6/10 | epoch 1 | training batch 180 | loss 0.67675
Fold 6/10 | epoch 1/50 | train 0.67605 | validation 0.66031 | best 1 | patience 0/10 | 4.2s
Fold 6/10 | epoch 2 | training batch 20 | loss 0.65875
Fold 6/10 | epoch 2 | training batch 40 | loss 0.65732
Fold 6/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
48,0.4735390238153438,9600,0.5561253289363043,0.7990316981821504,41,7
49,0.4739173559161524,9600,0.5611931900053015,0.8000664038619643,41,8
50,0.46906089736148715,9600,0.5657127848717244,0.7943412666624506,41,9


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
6,0.7223628691983123,0.7877973884508453,0.7982838868431547,0.562210261670849,41,0.5471201622273839,9600,1067,1185


### 6.7 Fold 7/10

In [13]:
run_phase('train', fold=7)
show_csv(RESULTS/'fold07'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold07'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 7
Fold 7/10 | epoch 1 | training batch 20 | loss 0.69503
Fold 7/10 | epoch 1 | training batch 40 | loss 0.69138
Fold 7/10 | epoch 1 | training batch 60 | loss 0.69019
Fold 7/10 | epoch 1 | training batch 80 | loss 0.68818
Fold 7/10 | epoch 1 | training batch 100 | loss 0.68573
Fold 7/10 | epoch 1 | training batch 120 | loss 0.68404
Fold 7/10 | epoch 1 | training batch 140 | loss 0.68187
Fold 7/10 | epoch 1 | training batch 160 | loss 0.68166
Fold 7/10 | epoch 1 | training batch 180 | loss 0.67918
Fold 7/10 | epoch 1/50 | train 0.67746 | validation 0.65992 | best 1 | patience 0/10 | 5.0s
Fold 7/10 | epoch 2 | training batch 20 | loss 0.66174
Fold 7/10 | epoch 2 | training batch 40 | loss 0.66456
Fold 7/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
48,0.47632070009907085,9600,0.561708682702877,0.7912617436459586,48,0
49,0.4744696165435016,9600,0.5722509123168488,0.7905520304122662,48,1
50,0.4701390761571626,9600,0.5678768962370311,0.7908225646647131,48,2


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
7,0.7350210970464135,0.7991730664053598,0.7915318597759312,0.5565395371320379,48,0.561708682702877,9600,1067,1185


### 6.8 Fold 8/10

In [14]:
run_phase('train', fold=8)
show_csv(RESULTS/'fold08'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold08'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 8
Fold 8/10 | epoch 1 | training batch 20 | loss 0.69637
Fold 8/10 | epoch 1 | training batch 40 | loss 0.69293
Fold 8/10 | epoch 1 | training batch 60 | loss 0.69145
Fold 8/10 | epoch 1 | training batch 80 | loss 0.68903
Fold 8/10 | epoch 1 | training batch 100 | loss 0.68683
Fold 8/10 | epoch 1 | training batch 120 | loss 0.68307
Fold 8/10 | epoch 1 | training batch 140 | loss 0.68083
Fold 8/10 | epoch 1 | training batch 160 | loss 0.67996
Fold 8/10 | epoch 1 | training batch 180 | loss 0.67716
Fold 8/10 | epoch 1/50 | train 0.67623 | validation 0.66848 | best 1 | patience 0/10 | 4.6s
Fold 8/10 | epoch 2 | training batch 20 | loss 0.65996
Fold 8/10 | epoch 2 | training batch 40 | loss 0.65471
Fold 8/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
48,0.48183769018699724,9600,0.5589575378718282,0.7899723141570223,44,4
49,0.47984671701366705,9600,0.5634288832531315,0.7875410193168484,44,5
50,0.47697465199356276,9600,0.5633432447295827,0.7888480159650343,44,6


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
8,0.7274261603375527,0.7989551524543093,0.7863557263681432,0.5531422643218865,44,0.554925954330716,9600,1067,1185


### 6.9 Fold 9/10

In [15]:
run_phase('train', fold=9)
show_csv(RESULTS/'fold09'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold09'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 9
Fold 9/10 | epoch 1 | training batch 20 | loss 0.69591
Fold 9/10 | epoch 1 | training batch 40 | loss 0.69265
Fold 9/10 | epoch 1 | training batch 60 | loss 0.69118
Fold 9/10 | epoch 1 | training batch 80 | loss 0.68935
Fold 9/10 | epoch 1 | training batch 100 | loss 0.68637
Fold 9/10 | epoch 1 | training batch 120 | loss 0.68456
Fold 9/10 | epoch 1 | training batch 140 | loss 0.68255
Fold 9/10 | epoch 1 | training batch 160 | loss 0.68128
Fold 9/10 | epoch 1 | training batch 180 | loss 0.67855
Fold 9/10 | epoch 1/50 | train 0.67790 | validation 0.66171 | best 1 | patience 0/10 | 4.7s
Fold 9/10 | epoch 2 | training batch 20 | loss 0.66182
Fold 9/10 | epoch 2 | training batch 40 | loss 0.66457
Fold 9/10 | epoch 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
48,0.4704385995088766,9600,0.5097603963263815,0.8282107496960881,41,7
49,0.46821791275093955,9600,0.5146895679448851,0.8244302970255286,41,8
50,0.46825314794356626,9600,0.5028511295837002,0.8330627990808861,50,0


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
9,0.7426160337552743,0.8075691627546603,0.8142133826028408,0.5390193037845917,50,0.5028511295837002,9600,1067,1185


### 6.10 Fold 10/10

In [16]:
run_phase('train', fold=10)
show_csv(RESULTS/'fold10'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold10'/'metrics.json').read_text())])

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py train --run-name msi_seed0 --fold 10
Fold 10/10 | epoch 1 | training batch 20 | loss 0.69672
Fold 10/10 | epoch 1 | training batch 40 | loss 0.69326
Fold 10/10 | epoch 1 | training batch 60 | loss 0.69133
Fold 10/10 | epoch 1 | training batch 80 | loss 0.68925
Fold 10/10 | epoch 1 | training batch 100 | loss 0.68483
Fold 10/10 | epoch 1 | training batch 120 | loss 0.68354
Fold 10/10 | epoch 1 | training batch 140 | loss 0.68101
Fold 10/10 | epoch 1 | training batch 160 | loss 0.68065
Fold 10/10 | epoch 1 | training batch 180 | loss 0.67918
Fold 10/10 | epoch 1/50 | train 0.67783 | validation 0.66911 | best 1 | patience 0/10 | 4.2s
Fold 10/10 | epoch 2 | training batch 20 | loss 0.65378
Fold 10/10 | epoch 2 | training batch 40 | loss 0.65917
Fold 

epoch,train_loss,training_samples,validation_loss,validation_auroc,best_epoch,epochs_without_improvement
32,0.5019643153063953,9600,0.5807791417630453,0.7744991602897878,24,8
33,0.4984713850232462,9600,0.5742032918956868,0.7774926042259558,24,9
34,0.4980084023748835,9600,0.5760832431725285,0.7761293926681705,24,10


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
10,0.6978902953586498,0.7622159997265393,0.7607494946996267,0.5849623748521765,24,0.5697247479268068,9600,1067,1185


## 7. Summarize all ten folds

Verify exactly one held-out prediction per labeled pair, then report fold metrics and mean ± SD. Do not report a ten-fold average before every fold finishes. This SD is **between folds**, rather than the paper's between-repeat SD.


In [17]:
run_phase('summarize')
show_csv(RESULTS/'fold_results.csv')
show_csv(RESULTS/'summary.csv')

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py summarize --run-name msi_seed0
[
  {
    "metric": "accuracy",
    "mean": 0.7251096832952661,
    "SD": 0.014538200712197604,
    "n": 10
  },
  {
    "metric": "auroc",
    "mean": 0.7919644962502151,
    "SD": 0.01403167635139362,
    "n": 10
  },
  {
    "metric": "aupr",
    "mean": 0.7943758617376557,
    "SD": 0.015761281710558135,
    "n": 10
  }
]


fold,accuracy,auroc,aupr,test_loss,best_epoch,validation_loss,n_train,n_validation,n_test
1,0.7166947723440135,0.7849162090607397,0.8002599734555893,0.5597959087429722,26,0.563624073661368,9599,1067,1186
2,0.7335581787521079,0.8046589070351403,0.8126740565187143,0.5416197495355975,42,0.5628382853514252,9599,1067,1186
3,0.7265822784810126,0.7954884690761588,0.7984583187435125,0.5477907719994396,30,0.5388681810373666,9600,1067,1185
4,0.7071729957805907,0.7774087325099129,0.7803828818155927,0.57069426749829,29,0.6060429361230766,9600,1067,1185
5,0.7417721518987341,0.8014618750284855,0.8008490365534515,0.555680707134778,50,0.5357393633012137,9600,1067,1185
6,0.7223628691983123,0.7877973884508453,0.7982838868431547,0.562210261670849,41,0.5471201622273839,9600,1067,1185
7,0.7350210970464135,0.7991730664053598,0.7915318597759312,0.5565395371320379,48,0.561708682702877,9600,1067,1185
8,0.7274261603375527,0.7989551524543093,0.7863557263681432,0.5531422643218865,44,0.554925954330716,9600,1067,1185
9,0.7426160337552743,0.8075691627546603,0.8142133826028408,0.5390193037845917,50,0.5028511295837002,9600,1067,1185
10,0.6978902953586498,0.7622159997265393,0.7607494946996267,0.5849623748521765,24,0.5697247479268068,9600,1067,1185


metric,mean,SD,n
accuracy,0.7251096832952661,0.014538200712197604,10
auroc,0.7919644962502151,0.01403167635139362,10
aupr,0.7943758617376557,0.015761281710558135,10


## 8. Compare with the paper's SEAL baseline

Inspect `local_minus_paper`; positive values indicate higher local metrics. Compare all three measures with Figure 2a SEAL rows. One local repeat is not the paper's ten-repeat protocol.


In [18]:
run_phase('compare')
show_csv(RESULTS/'paper_comparison.csv')
print('Results: ', RESULTS)
print('Each best-fold model was saved before test evaluation.')

Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/src/run_seal.py compare --run-name msi_seed0
[
  {
    "metric": "accuracy",
    "local_seed0_10fold_mean": 0.7251096832952661,
    "paper_10repeat_mean": 0.7294390552509488,
    "local_minus_paper": -0.004329371955682615,
    "local_fold_SD": 0.014538200712197604,
    "paper_repeat_SD": 0.013069738112634448
  },
  {
    "metric": "auroc",
    "local_seed0_10fold_mean": 0.7919644962502151,
    "paper_10repeat_mean": 0.7936975331042188,
    "local_minus_paper": -0.0017330368540036112,
    "local_fold_SD": 0.01403167635139362,
    "paper_repeat_SD": 0.013848399089751065
  },
  {
    "metric": "aupr",
    "local_seed0_10fold_mean": 0.7943758617376557,
    "paper_10repeat_mean": 0.7937583979948021,
    "local_minus_paper": 0.0006174637428536522,
    "local_fold_SD":

metric,local_seed0_10fold_mean,paper_10repeat_mean,local_minus_paper,local_fold_SD,paper_repeat_SD
accuracy,0.7251096832952661,0.7294390552509488,-0.004329371955682615,0.014538200712197604,0.013069738112634448
auroc,0.7919644962502151,0.7936975331042188,-0.0017330368540036112,0.01403167635139362,0.013848399089751065
aupr,0.7943758617376557,0.7937583979948021,0.0006174637428536522,0.015761281710558135,0.012199522120737346


Results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/04.SEAL_reproduction/results/msi_seed0
The best model for each fold was saved before test evaluation.


## 9. Outputs and future controls

- Per-fold checkpoints, validation history and predictions: `results/<RUN_NAME>/foldXX/`.
- Pair/fold IDs: `fold_assignments.csv`; inner partitions: `partitions.json`.
- All held-out predictions: `oof_predictions.csv`; means: `summary.csv`; reference comparison: `paper_comparison.csv`.
- Add future SEAL controls as sections in this notebook. This baseline does not automatically add tuning or seeds.

Sources: [DREAMwalk Supplementary Method 1](https://www.nature.com/articles/s41467-023-39301-y#Sec15), [SEAL](https://github.com/muhanzhang/SEAL/tree/ca1f019a15fb0c21796042165b4e6bee73981dd3), [DGCNN](https://github.com/muhanzhang/pytorch_DGCNN/tree/50f504131ca66382f3f078ae98e59aa3ae35b795).
